# Digit Prediction

We try to identify Digits from 28x28 pixel image in greyscale.

In [21]:
%pip install scikit-learn
%pip install pandas 
%pip install seaborn
%pip install tensorflow
%pip install xgboost
%pip install lightgbm
%pip install pygad
%pip install scikeras


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [22]:
# Imports
import pandas as pd


import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
from sklearn import preprocessing, svm 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn import datasets, linear_model
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score,root_mean_squared_log_error


In [23]:
train_url = './digits/train.csv'
test_url = './digits/test.csv'
train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

print("Training dataset shape", train.shape)
print("Test dataset shape", test.shape)


Training dataset shape (42000, 785)
Test dataset shape (28000, 784)


In [24]:
# Split train dataset 
feature_cols = train.columns[1:]
print("Features num=", len(feature_cols), feature_cols)
small_train = train.sample(frac=0.8,random_state=200)

X=small_train[feature_cols]
Y=small_train.loc[:, ['label']]
Xval=train.drop(X.index)[feature_cols]
Yval=train.drop(X.index).loc[:, ['label']]

print("Training dataset shape", X.shape, Y.shape)
print("Validation dataset shape", Xval.shape, Yval.shape)


Features num= 784 Index(['pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=784)
Training dataset shape (33600, 784) (33600, 1)
Validation dataset shape (8400, 784) (8400, 1)


In [25]:
def build_decision_tree():
    print("Decision tree...")
    # Writing a small decision tree classifier
    from sklearn import tree
    clf = tree.DecisionTreeClassifier()
    clf = clf.fit(X, Y)
    #tree.plot_tree(clf)
    return clf

In [26]:
def build_svc():
    print("SVC...")
    from sklearn import svm
    from sklearn.svm import LinearSVC
    clf = svm.SVC(kernel='rbf', probability=True)
    clf.fit(X, Y)
    return clf

In [27]:
def build_nearest_centroid():
    print("NearestCentroid...")
    from sklearn.neighbors import NearestCentroid
    clf = NearestCentroid()
    clf.fit(X, Y)
    return clf

In [28]:

from sklearn.neighbors import KNeighborsClassifier
def build_knn(n_neighbors: int = 2):
    print("KNeighborsClassifier...")

    clf =  KNeighborsClassifier(n_neighbors=n_neighbors)  
    clf.fit(X, Y)
    return clf

In [29]:

from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import Pipeline
def build_complex_knn(n_neighbors: int = 3):
    print("ComplexKNeighborsClassifier...")
    nca = NeighborhoodComponentsAnalysis(random_state=42)

    knn = KNeighborsClassifier(n_neighbors=n_neighbors)

    nca_pipe = Pipeline([('nca', nca), ('knn', knn)])

    nca_pipe.fit(X, Y)
    return nca_pipe

In [30]:
from sklearn.ensemble import AdaBoostClassifier
def build_ada(estimators=200):
    clf = AdaBoostClassifier(n_estimators=estimators, algorithm="SAMME")
    clf.fit(X,Y)
    return clf

In [31]:
def build_bagging_classifier():
    from sklearn.ensemble import BaggingClassifier
    model = BaggingClassifier(n_estimators=20)
    model.fit(X, Y)
    return model
    

In [32]:
from sklearn.ensemble import HistGradientBoostingClassifier


def build_HistGradientBoostingClassifier(iter=20):
    print("build_HistGradientBoostingClassifier...")
    clf = HistGradientBoostingClassifier(max_iter=iter).fit(X, Y)
    return clf

In [33]:
# Create a function that evaluates a classifier and produce the metrics
from sklearn.metrics import accuracy_score,f1_score,recall_score, precision_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

def evaluate_model(model, modelName: str):
    print("Prediction for ", modelName)
    prediction = model.predict(Xval)
    acc = accuracy_score( Yval, prediction)
    print(modelName, "Accuracy", acc)
    
    #matrix = confusion_matrix(Yval, prediction)
    #print(matrix)
    #disp = ConfusionMatrixDisplay(confusion_matrix=matrix) 
    # Then just plot it: 
    #disp.plot() 
    # And show it: 
    #plt.show()    
    
    print(modelName, "F1", f1_score(Yval, prediction, average="macro"))
    print(modelName, "Recall", recall_score(Yval, prediction, average="macro"))
    print(modelName, "Precision", precision_score(Yval, prediction, average="macro"))
    

In [34]:
# Test Decision tree accuracy
k_model = build_knn()
evaluate_model(k_model, "knn")


KNeighborsClassifier...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Prediction for  knn
knn Accuracy 0.9610714285714286
knn F1 0.960585485798733
knn Recall 0.9602799685182963
knn Precision 0.9619758704314505


In [36]:

nc_model = build_nearest_centroid()
evaluate_model(nc_model, "nearest_centroid")


NearestCentroid...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Prediction for  nearest_centroid
nearest_centroid Accuracy 0.8086904761904762
nearest_centroid F1 0.8069959351056537
nearest_centroid Recall 0.8052873849247553
nearest_centroid Precision 0.8125884675387727


In [ ]:

#svc_model = build_svc()
hgb_model = build_HistGradientBoostingClassifier()
evaluate_model(hgb_model, "hgb_model")


In [37]:

ada_model = build_ada()

evaluate_model(ada_model, "ada")


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:

bag_model = build_bagging_classifier()
evaluate_model(bag_model, "bag_model")


In [ ]:

d_model = build_decision_tree()
evaluate_model(d_model, "decision_tree")


In [ ]:

cknn_model = build_complex_knn()
evaluate_model(cknn_model, "complex_knn")


In [ ]:

from sklearn.ensemble import VotingClassifier
eclf = VotingClassifier(estimators=[('dt', d_model), ('knn', k_model), ('nc', nc_model), ('hgb', hgb_model), ('ada', ada_model)])
eclf.fit(X,Y)
evaluate_model(eclf, "eclf")

In [ ]:

from sklearn.ensemble import VotingClassifier
eclf = VotingClassifier(estimators=[('dt', d_model), ('knn', k_model), ('nc', nc_model), ('ada', ada_model)])
eclf.fit(X,Y)
evaluate_model(eclf, "eclf")

In [35]:
def generate_prediction_test(model):
  kaggle_predict = model.predict(test)  
  submission_df = pd.DataFrame()
  submission_df['Label'] = kaggle_predict
  submission_df['ImageId'] = range(1, len(submission_df)+1 )
  submission_df.to_csv('digit_submission.csv', index=False) 
  
generate_prediction_test(k_model) 